>Feature engineering transforms raw transaction data into behavioral signals that help reveal abnormal patterns. In anomaly detection, the quality of features often matters more than the choice of model.
>Effective fraud detection focuses on behavioral rather than static features, with time playing a central role in capturing changes in activity. At the same time, care must be taken to avoid information leakage, as it can lead to misleading model performance.

# PaySim

## Time-based Transaction Count Features

In [2]:
import pandas as pd

paysim = pd.read_csv("../content/paysim.csv")

In [ ]:
# sort step feature
paysim = paysim.sort_values(["nameOrig", "step"]).reset_index(drop=True)

In [3]:
paysim.groupby("nameOrig")["step"].is_monotonic_increasing.all()

np.True_

In [13]:
import numpy as np

groups = paysim.groupby("nameOrig").indices
steps_all = paysim["step"].values

windows = {
    "24h": 24,
    "72h": 72,
    "7d": 168
}

for name, window in windows.items():
    tx_count = np.zeros(len(paysim), dtype=np.int32)

    for idx in groups.values():
        steps = steps_all[idx]
        j = 0

        for i in range(len(steps)):
            while steps[j] < steps[i] - window:
                j += 1
            tx_count[idx[i]] = i - j + 1

    paysim[f"tx_count_last_{name}"] = tx_count


In [14]:
paysim["tx_count_last_24h"].isna().sum()

np.int64(0)

In [11]:
paysim["tx_count_last_24h"].describe()


,tx_count_last_24h
count,6.362620e+06
mean,1.000181e+00
std,1.345454e-02
min,1.000000e+00
25%,1.000000e+00
50%,1.000000e+00
75%,1.000000e+00
max,2.000000e+00


In [15]:
paysim["tx_count_last_72h"].isna().sum()

np.int64(0)

In [16]:
paysim["tx_count_last_72h"].describe()

,tx_count_last_72h
count,6.362620e+06
mean,1.000440e+00
std,2.096200e-02
min,1.000000e+00
25%,1.000000e+00
50%,1.000000e+00
75%,1.000000e+00
max,2.000000e+00


In [17]:
paysim["tx_count_last_7d"].isna().sum()

np.int64(0)

In [18]:
paysim["tx_count_last_7d"].describe()


,tx_count_last_7d
count,6.362620e+06
mean,1.000882e+00
std,2.970701e-02
min,1.000000e+00
25%,1.000000e+00
50%,1.000000e+00
75%,1.000000e+00
max,3.000000e+00


In [20]:
paysim["tx_count_total_user"] = (
    paysim
    .groupby("nameOrig")["step"]
    .transform("count")
)

To capture short-term and mid-term behavioral patterns, several time-based transaction count features were engineered using sliding temporal windows. Specifically, the number of transactions performed by each user within the last 24 hours, 72 hours, and 7 days was computed based on the step variable, which represents elapsed hours in the PaySim simulation. These features aim to model bursty or unusually frequent transaction behavior, which is often indicative of fraudulent activity.

In addition, a global baseline feature (tx_count_total_user) was introduced to represent the total number of transactions performed by each user throughout the entire simulation. This feature provides contextual information about a user's overall activity level and allows the model to distinguish between inherently low-activity users and sudden deviations from typical behavior when combined with time-based counts.

## Time-based Transaction Amount Features